# NLP Lab Assignment — Experiment 3
## Title: Implement NLP Pre-processing Tasks
**Case Study Domain:** Terms and Conditions Summarizer (Amazon & Alibaba Legal Agreements)

### Objective:
To implement comprehensive text pre-processing tasks on contractual legal documents including HTML/special-character removal, whitespace cleaning, sentence segmentation, contraction expansion, lowercasing, number/currency handling, stop-word removal with negation retention, phrase extraction, tokenization, and script validation.

### 1. Ingestion of Raw Contract Excerpt

In [1]:
import re
import unicodedata
from bs4 import BeautifulSoup
import contractions
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
import pandas as pd

# Dirty HTML sample containing real legal clause with contractions, numbers, currencies, and special characters
raw_legal_sample = """
<div class="clause">
    <h3>Section 14.2: Limitation of Liability &amp; Disclaimers</h3>
    <p>We <b>can't</b> guarantee that Amazon's or Alibaba's services won't be interrupted. 
    Under <i>no circumstances</i> shall liability exceed <b>$1,000.00 USD</b> or 100% of fees paid in 2026. 
    You mustn't copy, reproduce, or resell any listings &copy; 2026! Contact: <a href="mailto:legal@terms.com">legal@terms.com</a>.</p>
</div>
"""

print("--- Raw Legal Sample Text ---")
print(raw_legal_sample)

--- Raw Legal Sample Text ---

<div class="clause">
    <h3>Section 14.2: Limitation of Liability &amp; Disclaimers</h3>
    <p>We <b>can't</b> guarantee that Amazon's or Alibaba's services won't be interrupted. 
    Under <i>no circumstances</i> shall liability exceed <b>$1,000.00 USD</b> or 100% of fees paid in 2026. 
    You mustn't copy, reproduce, or resell any listings &copy; 2026! Contact: <a href="mailto:legal@terms.com">legal@terms.com</a>.</p>
</div>



### 2. Comprehensive Pre-processing Pipeline Implementation

In [2]:
# Task 1: HTML Tag Removal
def remove_html(text):
    return BeautifulSoup(text, "html.parser").get_text(separator=" ")

# Task 2: Script / Unicode Validation (Clean Unicode anomalies)
def validate_and_clean_script(text):
    # Normalize unicode (NFKD) and filter standard ASCII / valid characters
    normalized = unicodedata.normalize('NFKD', text)
    return normalized.encode('ascii', 'ignore').decode('utf-8', 'ignore')

# Task 3: Contraction Expansion
def expand_contractions_text(text):
    return contractions.fix(text)

# Task 4: URL & Email Removal
def remove_urls_emails(text):
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", " ", text)
    return text

# Task 5: Number & Currency Normalization / Handling
def normalize_legal_numbers(text):
    # Standardize monetary figures and percentages for consistent representation
    text = re.sub(r"\$(\d+(?:,\d+)*(?:\.\d+)?)", r" CURRENCY_USD_\1 ", text)
    text = re.sub(r"(\d+)%", r" \1_PERCENT ", text)
    return text

# Task 6: Special Characters & Whitespace Cleaning
def clean_special_chars_whitespace(text):
    text = re.sub(r"[^a-zA-Z0-9_$.%\s-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

# Task 7: Sentence Segmentation (Clause Segmentation)
def segment_clauses(text):
    return sent_tokenize(text)

# Task 8: Domain-Specific Stop-Word Removal (Preserving Negation)
def remove_stopwords_preserve_negation(tokens):
    standard_stops = set(stopwords.words("english"))
    legal_negations = {"not", "no", "never", "without", "neither", "nor", "cannot", "mustn't", "none"}
    active_stops = standard_stops - legal_negations
    return [w for w in tokens if w.lower() not in active_stops]

# Execute step-by-step pipeline
step1 = remove_html(raw_legal_sample)
step2 = validate_and_clean_script(step1)
step3 = expand_contractions_text(step2)
step4 = remove_urls_emails(step3)
step5 = normalize_legal_numbers(step4)
step6 = clean_special_chars_whitespace(step5)
step7_sents = segment_clauses(step6)

print("=== Pipeline Preprocessing Results ===")
print(f"1. Expanded & Cleaned Text:\n   -> {step6}\n")
print(f"2. Segmented Sentences ({len(step7_sents)} sentences):")
for i, s in enumerate(step7_sents, 1):
    print(f"   [{i}] {s}")

# Task 9 & 10: Tokenization & Phrase Extraction on Segmented Sentence
words = word_tokenize(step7_sents[0])
filtered_words = remove_stopwords_preserve_negation(words)

print("\n3. Word Tokenisation:", words)
print("4. Filtered Tokens (Negation Preserved):", filtered_words)

=== Pipeline Preprocessing Results ===
1. Expanded & Cleaned Text:
   -> Section 14.2 Limitation of Liability Disclaimers We cannot guarantee that Amazon s or Alibaba s services will not be interrupted. Under no circumstances shall liability exceed CURRENCY_USD_1 000.00 USD or 100_PERCENT of fees paid in 2026. You must not copy reproduce or resell any listings 2026 Contact .

2. Segmented Sentences (3 sentences):
   [1] Section 14.2 Limitation of Liability Disclaimers We cannot guarantee that Amazon s or Alibaba s services will not be interrupted.
   [2] Under no circumstances shall liability exceed CURRENCY_USD_1 000.00 USD or 100_PERCENT of fees paid in 2026.
   [3] You must not copy reproduce or resell any listings 2026 Contact .

3. Word Tokenisation: ['Section', '14.2', 'Limitation', 'of', 'Liability', 'Disclaimers', 'We', 'can', 'not', 'guarantee', 'that', 'Amazon', 's', 'or', 'Alibaba', 's', 'services', 'will', 'not', 'be', 'interrupted', '.']
4. Filtered Tokens (Negation Preser